# Track A — Apply the Reading Framework
Run each block, look at the plot, and answer in the **answer** cell. Bring your answers back to the group.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
plt.rcParams.update({"figure.figsize": (10, 3.8), "axes.titleweight": "bold"})
BLUE, ORANGE, GREEN = "#1f6feb", "#fb8500", "#2a9d8f"

DATA = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data"
energy = pd.read_csv(DATA / "energy_dataset.csv"); energy["time"] = pd.to_datetime(energy["time"], utc=True)
weather = pd.read_csv(DATA / "weather_features.csv"); weather["dt_iso"] = pd.to_datetime(weather["dt_iso"], utc=True)
weather["city_name"] = weather["city_name"].str.strip()
load = energy.set_index("time")["total load actual"]

## 1 · Frequency
Run the cell. **Q:** what is the time step? which lags will be meaningful?

In [ ]:
energy["time"].diff().value_counts().head()

> **Answer:** 

## 2 · Trend & yearly cycle
Run the cells. **Q:** any level changes or abnormal periods? seasonal variation across the year?

In [ ]:
load.plot(lw=0.4, color=BLUE, title="Total load actual (MW)"); plt.show()

In [ ]:
load.groupby(load.index.month).mean().plot.bar(color=BLUE, title="Average load by month"); plt.show()

>  **Answer:** 

## 3 · Seasonality — daily & weekly
Run the cells. **Q:** is there a daily rhythm? a weekend effect?

In [ ]:
wk = load[(load.index >= pd.Timestamp("2017-01-09", tz="UTC")) & (load.index <= pd.Timestamp("2017-01-15 23:00", tz="UTC"))]
wk.plot(color=BLUE, title="One week (Jan 2017)"); plt.show()

In [ ]:
load.groupby(load.index.hour).mean().plot(marker="o", color=BLUE, title="Average load by hour"); plt.show()

In [ ]:
load.groupby(load.index.dayofweek).mean().plot.bar(color=BLUE, title="Average load by weekday (0=Mon)"); plt.show()

>  **Answer:** 

## 4 · Autocorrelation
Run the cell. **Q:** is the past predictive? which lag matters most besides t-1?

In [ ]:
for lag in [1, 24, 48, 168]:
    print(f"lag {lag:4d} h  ->  autocorr = {load.autocorr(lag):.2f}")

>  **Answer:** 

## 5 · External driver — temperature
Run the cell. **Q:** is demand related to temperature? linear or not?

In [ ]:
wmean = weather.drop_duplicates(subset=["dt_iso", "city_name"]).groupby("dt_iso")["temp"].mean()
df = energy.set_index("time").assign(temp_C=wmean - 273.15)
m = df.dropna(subset=["temp_C", "total load actual"])
curve = m.groupby(pd.cut(m["temp_C"], np.arange(-5, 46, 2.5)), observed=True)["total load actual"].mean()
plt.plot([i.mid for i in curve.index], curve.values, marker="o", color=ORANGE)
plt.title("Average load by temperature (°C)"); plt.xlabel("°C"); plt.ylabel("MW"); plt.show()
print("linear correlation:", round(m["temp_C"].corr(m["total load actual"]), 2))

>  **Answer:** 

## 6 · Before cleaning — a quick look
Run the cells. **Q:** any missing values (which columns)? what dominates the generation mix?

In [ ]:
miss = energy.isna().mean() * 100
print(miss[miss > 0].sort_values(ascending=False).round(1).to_string())

In [ ]:
gen = [c for c in energy.columns if c.startswith("generation")]
mix = energy[gen].mean(); mix = mix[mix > 0].sort_values()
mix.plot.barh(color=GREEN, title="Average generation by source (MW)"); plt.show()

>  **Answer:** 

## Reading notes → Part B
From your answers, fill the table:

| Observation | Next decision |
|---|---|
|  |  |
|  |  |
|  |  |
|  |  |